## 0. Setup (run first, every session)
**Before you start**
1. *Kaggle:* Settings -> Accelerator = **GPU T4/P100**, Internet = **On**; Add-ons -> Secrets -> add `HF_TOKEN` (a Hugging Face token with *write* access). *Colab:* Runtime -> GPU, and add `HF_TOKEN` under the key icon (Secrets), notebook access on.
2. Edit `HF_REPO` below (e.g. `yourname/atml-pa3-backup`). A **private** repo is created automatically on the first push.
3. A background loop pushes results + checkpoints to that repo every 10 minutes, and `pull` restores them in a fresh session. **If a session dies, re-run section 0 and then re-run the same training cell: it resumes from the last checkpoint.**

4. Put the assignment's `first_plot_questions.yaml` at `PA 3/data/first_plot_questions.yaml` on your **local** machine and run `python -m scripts.sync_hub push --repo <HF_REPO> --task task1` once from `PA 3/` - it is included in the backup, so section 0's `pull` brings it here.
5. Budget (estimates, not measured): one full-FT run is a few hours on a T4; LoRA runs are faster. Run **one method per session** if you are unsure. Kaggle's output disk is 20 GB: one full-FT run is about 12 GB, so after each full-FT run is pushed, delete its `recovery_*.pt` (cell provided).

In [ ]:
import os, sys, subprocess, threading, time
HF_REPO  = "YOUR_USERNAME/atml-pa3-backup"      # <- edit
REPO_URL = "https://github.com/ZainFarhan-Work/ATML.git"
BRANCH   = "main"

ON_KAGGLE = os.path.exists("/kaggle")
ROOT = "/kaggle/working" if ON_KAGGLE else "/content"
if ON_KAGGLE:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
else:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

if not os.path.exists(f"{ROOT}/ATML"):
    subprocess.run(["git", "clone", "-b", BRANCH, REPO_URL, f"{ROOT}/ATML"], check=True)
else:
    subprocess.run(["git", "-C", f"{ROOT}/ATML", "pull"], check=False)
os.chdir(f"{ROOT}/ATML/PA 3")
print(os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m scripts.check_environment

In [ ]:
# restore anything saved by a previous session (harmless on the first run)
r = subprocess.run([sys.executable, "-m", "scripts.sync_hub", "pull", "--repo", HF_REPO, "--task", "task1"])
print("pull exit code:", r.returncode)

# background backup every 10 min (and call sync_now() manually after each section)
def sync_now():
    subprocess.run([sys.executable, "-m", "scripts.sync_hub", "push", "--repo", HF_REPO, "--task", "task1"])
def _loop():
    while True:
        time.sleep(600)
        try: sync_now()
        except Exception as e: print("sync failed:", e)
threading.Thread(target=_loop, daemon=True).start()

## 1.0 Smoke test (5 updates, ~minutes) - do this before spending real compute
Reports seconds/update and peak GPU memory for full FT. If it runs out of memory, set `micro_batch: 1` and `grad_accum: 32` in `configs/task1_llm.yaml` (same effective batch of 32).

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method full_ft --max-updates 5
import json; print(json.load(open("results/task1/full_ft/resources.json")))
# remove the smoke-test artifacts so the real run starts clean
!rm -rf results/task1/full_ft checkpoints/task1/full_ft

## 1.1 Full fine-tuning (lr 2e-5, lambda 0)
Resumable: if the session dies, re-run section 0 and this cell.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method full_ft --resume
sync_now()

## 1.2 Full FT + L2-SP (lambda 1e-3)
Starts again from the original checkpoint. theta_0 is kept on CPU (`ref_device: cpu`) to fit 16 GB.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method full_ft_l2sp --resume
sync_now()

### (Kaggle disk) free space once a full-FT run is safely pushed
Only the final eval snapshots (`eval_625`, `eval_1250`) are needed afterwards; the recovery checkpoint is 7 GB.

In [ ]:
sync_now()   # make sure everything is on the Hub first
!rm -f checkpoints/task1/full_ft/recovery_*.pt checkpoints/task1/full_ft_l2sp/recovery_*.pt
!df -h . | tail -1

## 1.3 LoRA rank 4 and 16 (lr 2e-4)

In [ ]:
for m in ["lora4", "lora16"]:
    !PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method {m} --resume
    sync_now()

## 1.4 OPLoRA (r=16, k=128)
Computes FP32 SVD bases once (time/storage logged), then trains with effective factors.

In [ ]:
!PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True python -m task1_llm.train --method oplora16 --resume
sync_now()

## 1.5 Evaluation of the seven conditions (GSM8K, HellaSwag, BoolQ, ARC-Easy)
Slow part is GSM8K generation (1,319 questions, up to 1,024 tokens). Finished conditions are skipped on re-run, so this is safe to resume. Use `--only baseline full_ft_mid` etc. to split across sessions.

In [ ]:
!python -m task1_llm.evaluate
sync_now()

## 1.6 Diagnostics, qualitative generations, tables and figures

In [ ]:
!python -m task1_llm.diagnostics
!python -m task1_llm.qualitative
!python -m task1_llm.analyze
sync_now()

## Final: download what you need locally
Everything is also in your private HF repo. Locally run `python -m scripts.sync_hub pull --repo <HF_REPO> --task task1` from `PA 3/` (add `checkpoints/` only if you need them).